# Advanced Problems with Solutions
## Runtime Attributes, Dynamic Methods, `MethodType`, and Per-Instance Plugins

This notebook expands the topic into progressively harder, executable exercises.

### Main ideas
- Runtime attributes live on individual instances.
- A function placed directly on an instance is **not** automatically a bound method.
- `types.MethodType` can bind a function to exactly one object.
- Functions stored on a class participate in descriptor-based method binding.
- Instance attributes can shadow class attributes.
- Dynamic behavior can support plugin/capability systems, but it should be validated and contained.
- Copying, introspection, and serialization require extra care when bound methods are stored dynamically.

All examples use the Python standard library only.

## Setup

In [1]:
from types import MethodType
from inspect import signature, Parameter
from functools import wraps
from typing import Any, Callable
from dataclasses import dataclass
import copy
import json

---
# Problem 1 — Runtime attributes are isolated per instance

Create a `Device` class with a `serial` attribute. Create two devices and add `location`
to only one of them.

### Requirements
1. Inspect both instance dictionaries.
2. Safely read a missing attribute with `getattr`.
3. Prove with assertions that the runtime attribute is isolated.

In [2]:
class Device:
    def __init__(self, serial: str):
        self.serial = serial

d1 = Device("A-100")
d2 = Device("B-200")

d1.location = "Lab"

print("d1:", vars(d1))
print("d2:", vars(d2))
print("d2.location:", getattr(d2, "location", "<missing>"))

assert d1.location == "Lab"
assert "location" in d1.__dict__
assert "location" not in d2.__dict__
assert getattr(d2, "location", None) is None

d1: {'serial': 'A-100', 'location': 'Lab'}
d2: {'serial': 'B-200'}
d2.location: <missing>


### Solution insight

For ordinary objects with an instance `__dict__`, assigning `d1.location = "Lab"`
stores the name/value pair on `d1`. It does not update `d2` or the class.

---
# Problem 2 — Diagnose why an instance-stored function is not a method

Define:

```python
def greet(self):
    return f"Hello, {self.name}"
```

Assign it directly to an instance with `u.greet = greet`.

### Requirements
- Inspect the stored object.
- Show that `u.greet()` fails.
- Show that `u.greet(u)` works.
- Explain the missing automatic `self`.

In [3]:
class User:
    def __init__(self, name: str):
        self.name = name

def greet(self):
    return f"Hello, {self.name}"

u = User("Mira")
u.greet = greet

print("Stored object:", u.greet)
print("Type:", type(u.greet))

try:
    u.greet()
except TypeError as exc:
    print("Expected error:", exc)

print("Manual self:", u.greet(u))
assert u.greet(u) == "Hello, Mira"

Stored object: <function greet at 0x000001F6F55E0F40>
Type: <class 'function'>
Expected error: greet() missing 1 required positional argument: 'self'
Manual self: Hello, Mira


### Solution insight

A function stored directly in an **instance dictionary** is retrieved as that same
function object. Python does not automatically insert the instance as the first argument.

Automatic method binding normally happens when a function is retrieved from the **class**.

---
# Problem 3 — Bind a function to exactly one instance

Create two `User` objects. Bind `greet` only to the first object with `MethodType`.

### Requirements
- `u1.greet()` must work.
- `u2` must not receive the method.
- Inspect `__self__` and `__func__`.

In [4]:
u1 = User("Mira")
u2 = User("Ivan")

u1.greet = MethodType(greet, u1)

print(u1.greet())
print("bound self:", u1.greet.__self__)
print("underlying function:", u1.greet.__func__)

assert u1.greet() == "Hello, Mira"
assert u1.greet.__self__ is u1
assert u1.greet.__func__ is greet
assert not hasattr(u2, "greet")

Hello, Mira
bound self: <__main__.User object at 0x000001F6E53D0A50>
underlying function: <function greet at 0x000001F6F55E0F40>


---
# Problem 4 — Create a reusable `attach_method` helper

Implement a helper that safely installs a bound method on one object.

### Contract

```python
attach_method(obj, name, func, overwrite=False)
```

Rules:
- `name` must be a non-empty string.
- `func` must be callable.
- Existing attributes are protected unless `overwrite=True`.
- Return the newly attached bound method.

In [5]:
def attach_method(
    obj: Any,
    name: str,
    func: Callable[..., Any],
    *,
    overwrite: bool = False,
):
    if not isinstance(name, str) or not name.strip():
        raise ValueError("name must be a non-empty string")

    if not callable(func):
        raise TypeError("func must be callable")

    if hasattr(obj, name) and not overwrite:
        raise AttributeError(
            f"{type(obj).__name__} already has attribute {name!r}"
        )

    bound = MethodType(func, obj)
    setattr(obj, name, bound)
    return bound

In [6]:
class Robot:
    def __init__(self, robot_id: str):
        self.robot_id = robot_id

def identify(self):
    return f"Robot<{self.robot_id}>"

r = Robot("R-7")
attached = attach_method(r, "identify", identify)

print(r.identify())
assert r.identify() == "Robot<R-7>"
assert attached is r.identify

try:
    attach_method(r, "identify", identify)
except AttributeError as exc:
    print("Overwrite protected:", exc)

Robot<R-7>
Overwrite protected: Robot already has attribute 'identify'


### Best practice

Centralize dynamic mutation. A helper gives one place for validation, overwrite policy,
logging, instrumentation, and future security checks.

---
# Problem 5 — Override one class method for one instance

A `Product` class has a normal `price()` method. Override it on only one product.

### Requirements
- Normal products keep the class implementation.
- One special product uses a 20% discount.
- Deleting the instance-level override restores class behavior.

In [7]:
class Product:
    def __init__(self, name: str, base_price: float):
        self.name = name
        self.base_price = base_price

    def price(self) -> float:
        return self.base_price

def discounted_price(self) -> float:
    return self.base_price * 0.80

normal = Product("Keyboard", 100.0)
special = Product("Monitor", 250.0)

special.price = MethodType(discounted_price, special)

print("normal:", normal.price())
print("special:", special.price())
assert normal.price() == 100.0
assert special.price() == 200.0

del special.price

print("restored:", special.price())
assert special.price() == 250.0

normal: 100.0
special: 200.0
restored: 250.0


### Solution insight

The instance attribute shadows the class attribute for that object. After deleting the
instance attribute, normal attribute lookup finds the class method again.

---
# Problem 6 — Compare instance patching with class patching

Perform two experiments:

1. Add `private_ping` to only one instance.
2. Add `public_ping` to the class *after* two instances already exist.

Determine which instances see each method.

In [8]:
class Notifier:
    def __init__(self, name: str):
        self.name = name

n1 = Notifier("alpha")
n2 = Notifier("beta")

def private_ping(self):
    return f"private:{self.name}"

n1.private_ping = MethodType(private_ping, n1)

assert n1.private_ping() == "private:alpha"
assert not hasattr(n2, "private_ping")

def public_ping(self):
    return f"public:{self.name}"

Notifier.public_ping = public_ping

n3 = Notifier("gamma")

print(n1.public_ping())
print(n2.public_ping())
print(n3.public_ping())

assert n1.public_ping() == "public:alpha"
assert n2.public_ping() == "public:beta"
assert n3.public_ping() == "public:gamma"
assert n2.public_ping.__self__ is n2
assert n2.public_ping.__func__ is public_ping

public:alpha
public:beta
public:gamma


### Solution insight

Assigning a function to the class makes it available through normal class lookup.
Because functions implement the descriptor protocol, retrieving that function through an
instance produces a bound method.

---
# Problem 7 — Reproduce method binding with `__get__`

Functions are descriptors. Bind a function manually with:

```python
func.__get__(instance, type(instance))
```

Compare the result to `MethodType(func, instance)`.

In [9]:
class Account:
    def __init__(self, owner: str):
        self.owner = owner

def describe(self):
    return f"Account owner: {self.owner}"

a = Account("Nadia")

via_method_type = MethodType(describe, a)
via_descriptor = describe.__get__(a, type(a))

print(via_method_type())
print(via_descriptor())

assert via_method_type() == via_descriptor()
assert via_method_type.__self__ is a
assert via_descriptor.__self__ is a
assert via_method_type.__func__ is describe
assert via_descriptor.__func__ is describe

Account owner: Nadia
Account owner: Nadia


### Best practice

Use `MethodType` when the intent is explicit runtime binding. Use `__get__` directly mostly
for descriptor/framework internals or for learning how binding works.

---
# Problem 8 — Build a per-instance capability registry

Build an `Agent` whose instances can register different implementations under the same
capability name.

### API
- `register(name, func, replace=False)`
- `run(name, *args, **kwargs)`
- `capabilities()`
- `unregister(name)`

### Requirements
- Registries are isolated per instance.
- Duplicate registration is rejected unless replacement is requested.
- Missing capabilities raise a clear error.

In [10]:
class Agent:
    def __init__(self, name: str):
        self.name = name
        self._capabilities: dict[str, Callable[..., Any]] = {}

    def register(
        self,
        name: str,
        func: Callable[..., Any],
        *,
        replace: bool = False,
    ) -> None:
        if not isinstance(name, str) or not name:
            raise ValueError("capability name must be a non-empty string")
        if not callable(func):
            raise TypeError("capability must be callable")
        if name in self._capabilities and not replace:
            raise KeyError(f"capability {name!r} already registered")

        self._capabilities[name] = MethodType(func, self)

    def run(self, name: str, *args, **kwargs):
        try:
            method = self._capabilities[name]
        except KeyError as exc:
            raise LookupError(f"unknown capability: {name!r}") from exc
        return method(*args, **kwargs)

    def capabilities(self) -> tuple[str, ...]:
        return tuple(sorted(self._capabilities))

    def unregister(self, name: str) -> None:
        try:
            del self._capabilities[name]
        except KeyError as exc:
            raise LookupError(f"unknown capability: {name!r}") from exc

In [11]:
def add(self, x, y):
    return f"{self.name}: {x + y}"

def multiply(self, x, y):
    return f"{self.name}: {x * y}"

a1 = Agent("math-A")
a2 = Agent("math-B")

a1.register("calculate", add)
a2.register("calculate", multiply)

print(a1.run("calculate", 3, 4))
print(a2.run("calculate", 3, 4))

assert a1.run("calculate", 3, 4) == "math-A: 7"
assert a2.run("calculate", 3, 4) == "math-B: 12"
assert a1.capabilities() == ("calculate",)
assert a2.capabilities() == ("calculate",)

math-A: 7
math-B: 12


### Design improvement

A dedicated registry is often easier to reason about than injecting many arbitrary public
method names directly into an object's namespace.

---
# Problem 9 — Validate plugin signatures

A plugin may be callable but incompatible with your runtime contract.

For this exercise, accept functions that define exactly:

```text
self + N positional parameters
```

and reject variadic positional parameters (`*args`).

Implement:

```python
validate_plugin(func, required_user_args)
```

In [12]:
def validate_plugin(
    func: Callable[..., Any],
    required_user_args: int,
) -> None:
    if not callable(func):
        raise TypeError("plugin must be callable")
    if required_user_args < 0:
        raise ValueError("required_user_args must be >= 0")

    sig = signature(func)
    params = list(sig.parameters.values())

    if any(p.kind is Parameter.VAR_POSITIONAL for p in params):
        raise TypeError("*args is not allowed by this strict plugin contract")

    positional = [
        p
        for p in params
        if p.kind in (
            Parameter.POSITIONAL_ONLY,
            Parameter.POSITIONAL_OR_KEYWORD,
        )
    ]

    expected = 1 + required_user_args

    if len(positional) != expected:
        raise TypeError(
            f"expected exactly {expected} positional parameters "
            f"(self + {required_user_args} user args); got {sig}"
        )

In [13]:
def ok_transform(self, value):
    return f"{self.name}:{value}"

def bad_transform(self, x, y):
    return x + y

validate_plugin(ok_transform, 1)

try:
    validate_plugin(bad_transform, 1)
except TypeError as exc:
    print("Rejected:", exc)

Rejected: expected exactly 2 positional parameters (self + 1 user args); got (self, x, y)


### Advanced note

Real plugin systems often need default arguments, keyword-only parameters, or flexible
signatures. For those cases, `inspect.Signature.bind` / `bind_partial` can provide more
accurate call-compatibility checks.

---
# Problem 10 — Instrument a dynamically bound method with a decorator

Write `count_calls` so that each invocation stores a counter on the instance.

### Requirements
- Use `functools.wraps`.
- Counter name: `_call_count_<function_name>`.
- Attach the wrapped function to only one object.

In [14]:
def count_calls(func):
    @wraps(func)
    def wrapper(self, *args, **kwargs):
        counter_name = f"_call_count_{func.__name__}"
        current = getattr(self, counter_name, 0)
        setattr(self, counter_name, current + 1)
        return func(self, *args, **kwargs)
    return wrapper

class Worker:
    def __init__(self, name: str):
        self.name = name

def perform(self, task: str):
    return f"{self.name} performs {task}"

w = Worker("Ada")
instrumented = count_calls(perform)
w.perform = MethodType(instrumented, w)

print(w.perform("backup"))
print(w.perform("deploy"))
print("calls:", w._call_count_perform)

assert w._call_count_perform == 2
assert w.perform.__func__.__name__ == "perform"

Ada performs backup
Ada performs deploy
calls: 2


---
# Problem 11 — Discover the shallow-copy trap

A bound method remembers the object to which it is bound.

### Task
- Attach a dynamic method.
- Use `copy.copy`.
- Change the clone's data.
- Inspect the clone's copied bound method.
- Rebind it correctly.

In [15]:
class Profile:
    def __init__(self, name: str):
        self.name = name

def label(self):
    return f"Profile<{self.name}>"

original = Profile("Original")
original.label = MethodType(label, original)

cloned = copy.copy(original)
cloned.name = "Clone"

print("original:", original.label())
print("clone before fix:", cloned.label())
print("clone bound to original?", cloned.label.__self__ is original)

assert cloned.label.__self__ is original
assert cloned.label() == "Profile<Original>"

cloned.label = MethodType(cloned.label.__func__, cloned)

print("clone after fix:", cloned.label())
assert cloned.label.__self__ is cloned
assert cloned.label() == "Profile<Clone>"

original: Profile<Original>
clone before fix: Profile<Original>
clone bound to original? True
clone after fix: Profile<Clone>


### Best practice

If an object's instance dictionary stores bound methods, generic copying can preserve
bindings to the original object. If such objects must be copyable, implement a deliberate
copy/rebinding policy instead of relying blindly on shallow copy.

---
# Problem 12 — Implement a safe custom clone

Extend the previous idea with a class that can clone itself while rebinding all dynamic
methods stored directly on the instance.

### Requirements
- Copy ordinary instance data.
- Detect values of type `MethodType`.
- If a bound method is bound to the source object, rebind its underlying function to the clone.

In [16]:
class Cloneable:
    def clone(self):
        cls = type(self)
        new_obj = cls.__new__(cls)

        for name, value in self.__dict__.items():
            if isinstance(value, MethodType) and value.__self__ is self:
                rebound = MethodType(value.__func__, new_obj)
                setattr(new_obj, name, rebound)
            else:
                setattr(new_obj, name, copy.copy(value))

        return new_obj


class Config(Cloneable):
    def __init__(self, name: str, values: list[int]):
        self.name = name
        self.values = values

def summary(self):
    return f"{self.name}:{sum(self.values)}"

cfg = Config("A", [1, 2, 3])
cfg.summary = MethodType(summary, cfg)

cfg2 = cfg.clone()
cfg2.name = "B"
cfg2.values.append(10)

print(cfg.summary())
print(cfg2.summary())

assert cfg.summary.__self__ is cfg
assert cfg2.summary.__self__ is cfg2
assert cfg.summary() == "A:6"
assert cfg2.summary() == "B:16"

A:6
B:16


---
# Problem 13 — Prefer stable plugin identifiers for serialization

Serializing arbitrary dynamically bound methods can be fragile across processes, modules,
or notebook sessions.

Design a more stable approach:

- Maintain a global registry from plugin name to top-level function.
- Store only the plugin name in serializable state.
- Rebind after reconstruction.

In [17]:
PLUGIN_REGISTRY: dict[str, Callable[..., Any]] = {}

def register_plugin(name: str):
    def decorator(func):
        if name in PLUGIN_REGISTRY:
            raise KeyError(f"duplicate plugin name: {name!r}")
        PLUGIN_REGISTRY[name] = func
        return func
    return decorator


@register_plugin("upper")
def plugin_upper(self, text: str) -> str:
    return f"{self.prefix}{text.upper()}"


@register_plugin("reverse")
def plugin_reverse(self, text: str) -> str:
    return f"{self.prefix}{text[::-1]}"

In [18]:
class TextService:
    def __init__(self, prefix: str, plugin_name: str):
        self.prefix = prefix
        self.plugin_name = plugin_name
        self._bind_plugin()

    def _bind_plugin(self):
        try:
            func = PLUGIN_REGISTRY[self.plugin_name]
        except KeyError as exc:
            raise ValueError(
                f"unknown plugin {self.plugin_name!r}"
            ) from exc
        self.transform = MethodType(func, self)

    def to_json(self) -> str:
        # Serialize stable data, not the bound method object.
        return json.dumps({
            "prefix": self.prefix,
            "plugin_name": self.plugin_name,
        })

    @classmethod
    def from_json(cls, payload: str):
        data = json.loads(payload)
        return cls(**data)

In [19]:
service = TextService(">> ", "upper")
payload = service.to_json()
restored = TextService.from_json(payload)

print(payload)
print(restored.transform("hello"))

assert restored.transform("hello") == ">> HELLO"
assert restored.transform.__self__ is restored

{"prefix": ">> ", "plugin_name": "upper"}
>> HELLO


### Best practice

Persist configuration or plugin identifiers whenever possible. Reconstruct runtime bound
methods from known code rather than treating arbitrary method objects as durable data.

---
# Problem 14 — Add dependencies between capabilities

Build a small pipeline where a dynamically registered capability may call another
capability through the host object.

Register:
- `normalize(text)`
- `score(text)`

Then write `analyze(text)` that calls both capabilities.

In [20]:
class Pipeline:
    def __init__(self, name: str):
        self.name = name
        self._plugins: dict[str, Callable[..., Any]] = {}

    def register(self, name: str, func: Callable[..., Any]) -> None:
        if name in self._plugins:
            raise KeyError(name)
        self._plugins[name] = MethodType(func, self)

    def call(self, name: str, *args, **kwargs):
        try:
            method = self._plugins[name]
        except KeyError as exc:
            raise LookupError(f"missing plugin {name!r}") from exc
        return method(*args, **kwargs)

def normalize(self, text: str) -> str:
    return " ".join(text.lower().split())

def score(self, text: str) -> int:
    return len(text.split())

def analyze(self, text: str) -> dict[str, Any]:
    cleaned = self.call("normalize", text)
    return {
        "pipeline": self.name,
        "normalized": cleaned,
        "score": self.call("score", cleaned),
    }

pipe = Pipeline("text-v1")
pipe.register("normalize", normalize)
pipe.register("score", score)
pipe.register("analyze", analyze)

result = pipe.call("analyze", "  Dynamic   METHODS are Powerful  ")
print(result)

assert result == {
    "pipeline": "text-v1",
    "normalized": "dynamic methods are powerful",
    "score": 4,
}

{'pipeline': 'text-v1', 'normalized': 'dynamic methods are powerful', 'score': 4}


---
# Problem 15 — Support temporary method overrides with a context manager

Sometimes a test needs to override one object's method temporarily.

Implement a context manager that:
- stores whether the instance already had its own attribute,
- installs a bound replacement,
- restores the exact previous state on exit.

In [21]:
class temporary_method:
    def __init__(self, obj, name: str, func: Callable[..., Any]):
        self.obj = obj
        self.name = name
        self.func = func
        self.had_instance_attr = False
        self.old_value = None

    def __enter__(self):
        self.had_instance_attr = self.name in getattr(
            self.obj, "__dict__", {}
        )

        if self.had_instance_attr:
            self.old_value = self.obj.__dict__[self.name]

        setattr(self.obj, self.name, MethodType(self.func, self.obj))
        return getattr(self.obj, self.name)

    def __exit__(self, exc_type, exc, tb):
        if self.had_instance_attr:
            setattr(self.obj, self.name, self.old_value)
        else:
            delattr(self.obj, self.name)

        return False

In [22]:
class Greeter:
    def __init__(self, name: str):
        self.name = name

    def greet(self):
        return f"Hello {self.name}"

def excited(self):
    return f"HELLO {self.name.upper()}!!!"

g = Greeter("Lena")

print("before:", g.greet())

with temporary_method(g, "greet", excited):
    print("inside:", g.greet())
    assert g.greet() == "HELLO LENA!!!"

print("after:", g.greet())
assert g.greet() == "Hello Lena"
assert "greet" not in g.__dict__

before: Hello Lena
inside: HELLO LENA!!!
after: Hello Lena


---
# Problem 16 — Capstone: production-style per-instance strategy injection

Build a `DataProcessor` with configurable runtime behavior.

Each processor has:
- a `name`,
- a runtime `transform`,
- a runtime `validate`,
- an audit log.

### Requirements
1. Register `transform` and `validate` with signature checks.
2. Process one value by validating first, transforming second.
3. Keep strategies isolated per instance.
4. Log successful and failed operations.
5. Permit replacing a strategy intentionally.

In [23]:
class DataProcessor:
    def __init__(self, name: str):
        self.name = name
        self._transform = None
        self._validate = None
        self.audit: list[dict[str, Any]] = []

    @staticmethod
    def _check_one_arg_plugin(func):
        validate_plugin(func, required_user_args=1)

    def set_transform(self, func, *, replace: bool = False):
        self._check_one_arg_plugin(func)
        if self._transform is not None and not replace:
            raise RuntimeError("transform already configured")
        self._transform = MethodType(func, self)

    def set_validator(self, func, *, replace: bool = False):
        self._check_one_arg_plugin(func)
        if self._validate is not None and not replace:
            raise RuntimeError("validator already configured")
        self._validate = MethodType(func, self)

    def process(self, value):
        if self._transform is None:
            raise RuntimeError("transform is not configured")
        if self._validate is None:
            raise RuntimeError("validator is not configured")

        accepted = bool(self._validate(value))

        if not accepted:
            self.audit.append({
                "value": value,
                "status": "rejected",
            })
            raise ValueError(f"{value!r} rejected by validator")

        result = self._transform(value)

        self.audit.append({
            "value": value,
            "status": "ok",
            "result": result,
        })
        return result

In [24]:
def positive(self, value):
    return value > 0

def square(self, value):
    return value ** 2

def double(self, value):
    return value * 2

p1 = DataProcessor("square-positive")
p2 = DataProcessor("double-positive")

p1.set_validator(positive)
p1.set_transform(square)

p2.set_validator(positive)
p2.set_transform(double)

print(p1.process(5))
print(p2.process(5))

assert p1.process(3) == 9
assert p2.process(3) == 6

try:
    p1.process(-2)
except ValueError as exc:
    print("Rejected:", exc)

print("p1 audit:", p1.audit)
print("p2 audit:", p2.audit)

assert p1._transform.__self__ is p1
assert p2._transform.__self__ is p2
assert p1._transform.__func__ is square
assert p2._transform.__func__ is double

25
10
Rejected: -2 rejected by validator
p1 audit: [{'value': 5, 'status': 'ok', 'result': 25}, {'value': 3, 'status': 'ok', 'result': 9}, {'value': -2, 'status': 'rejected'}]
p2 audit: [{'value': 5, 'status': 'ok', 'result': 10}, {'value': 3, 'status': 'ok', 'result': 6}]


## Capstone extension

Replace `p2`'s transform at runtime with a cube operation, but leave `p1` unchanged.

In [25]:
def cube(self, value):
    return value ** 3

p2.set_transform(cube, replace=True)

print("p1:", p1.process(2))
print("p2:", p2.process(2))

assert p1.process(2) == 4
assert p2.process(2) == 8

p1: 4
p2: 8


---
# Challenge 17 — Implement `detach_method`

Write a helper that removes only an instance-level dynamic attribute and refuses to remove
a class-only method accidentally.

Then test it on a class that has both a normal class method and an injected method.

In [26]:
def detach_instance_attribute(obj: Any, name: str):
    instance_dict = getattr(obj, "__dict__", None)

    if instance_dict is None:
        raise TypeError(
            f"{type(obj).__name__} has no instance __dict__"
        )

    if name not in instance_dict:
        raise AttributeError(
            f"{name!r} is not stored directly on this instance"
        )

    value = instance_dict[name]
    delattr(obj, name)
    return value

In [27]:
class Demo:
    def normal(self):
        return "class method"

def dynamic(self):
    return "instance method"

demo = Demo()
demo.dynamic = MethodType(dynamic, demo)

removed = detach_instance_attribute(demo, "dynamic")

print("removed:", removed)
assert not hasattr(demo, "dynamic")
assert demo.normal() == "class method"

try:
    detach_instance_attribute(demo, "normal")
except AttributeError as exc:
    print("Protected class-only method:", exc)

removed: <bound method dynamic of <__main__.Demo object at 0x000001F6F554FB60>>
Protected class-only method: 'normal' is not stored directly on this instance


---
# Challenge 18 — Explain and prove bound-method identity behavior

Repeated access to a class-defined method generally creates a new bound-method wrapper.

Investigate:

```python
obj.method is obj.method
```

Then compare it to a bound method that was explicitly stored in the instance dictionary.

In [28]:
class IdentityDemo:
    def class_method(self):
        return "class"

def injected(self):
    return "injected"

x = IdentityDemo()

first = x.class_method
second = x.class_method

print("class lookup identity:", first is second)
print("same self:", first.__self__ is second.__self__)
print("same func:", first.__func__ is second.__func__)

assert first is not second
assert first.__self__ is second.__self__ is x
assert first.__func__ is second.__func__

x.injected = MethodType(injected, x)

stored_first = x.injected
stored_second = x.injected

print("stored instance identity:", stored_first is stored_second)
assert stored_first is stored_second

class lookup identity: False
same self: True
same func: True
stored instance identity: True


### Why?

For the class-defined function, descriptor access creates a bound-method object when the
attribute is retrieved. For the explicitly injected version, the bound method itself is
already stored in the instance dictionary, so repeated lookup returns that stored object.

---
# Design Exercise 19 — When should you *not* inject methods dynamically?

For each situation below, choose a better design where appropriate.

1. Every `Order` object always needs the same `total()` behavior.
2. A small fixed set of algorithms can be selected when an object is constructed.
3. A test temporarily needs one object to behave differently.
4. Runtime-loaded extensions must give different objects different capabilities.
5. The application's public API must be highly predictable for IDEs and static type checkers.

### Suggested solution

- **1:** Define a normal class method.
- **2:** Prefer composition / Strategy objects or constructor-injected callables.
- **3:** A controlled temporary override can be reasonable, especially in tests.
- **4:** A capability/plugin registry may be appropriate.
- **5:** Prefer explicit methods, protocols/interfaces, or typed strategy objects.

Dynamic binding is a specialized tool, not a default replacement for normal class design.

---
# Reference Pattern — Typed strategy injection without `MethodType`

Sometimes a callable field is simpler and more explicit than injecting a method.

In [29]:
@dataclass
class StrategyProcessor:
    transform: Callable[[int], int]

    def process(self, value: int) -> int:
        return self.transform(value)

sp1 = StrategyProcessor(transform=lambda x: x * 2)
sp2 = StrategyProcessor(transform=lambda x: x ** 2)

assert sp1.process(5) == 10
assert sp2.process(5) == 25

### Comparison

Use dynamic bound methods when the behavior truly benefits from receiving and operating on
the host object as `self`.

Use constructor-injected callables or Strategy objects when explicit composition is enough.
That usually improves readability, typing, testing, and tooling.

---
# Final checklist

When using runtime methods in real code:

- Prefer normal class methods for behavior shared by all instances.
- Prefer composition/Strategy for predictable configurable behavior.
- Use `MethodType` when a function really must become a method of one specific instance.
- Validate plugin names and signatures.
- Centralize registration rather than scattering `setattr` calls.
- Protect accidental overwrites.
- Keep a clear unregister/restore path.
- Remember that bound methods hold references to their bound objects.
- Be careful with shallow copies of objects storing bound methods.
- Persist plugin identifiers/configuration rather than arbitrary bound methods.
- Add tests that prove instance isolation.
- Document the dynamic API because static tools may not discover injected methods.